# **Manan Shah**
# *Theme: Gold and Reserves*

### Purpose:
This theme investigates how nations rely on gold as part of their total reserves and whether this reliance reflects differences in regional stability, income group, or macroeconomic strategy. The goal is to understand how the composition of reserves (with and without gold) signals a country's standing in socio-economic metrics.
___
## **Analytical Question 1:**
## *How does the proportion of gold in total reserves differ across countries and regions and what is the relationship between gold-share and income groups?*

### Rationale:
Understanding the distribution of gold reliance across global economies can reveal disparities in how nations hedge against currency risk or economic shocks. I predict that wealthier nations may prioritize liquidity over gold, while lower-income economies may maintain gold as a tangible asset of value stability.

### Task Abstraction:
- **Compute Derived Value**: What is the proportion of gold in total reserves across countries? (Gold Share)
- **Charactize Distribution**: How is gold-share distributed across income groups?
- **Compare**: Are there trends in gold-share between low income and high income countries?

___
## **Analytical Question 2:**
## *To what extent does a nation’s total reserves relate to social well-being indicators such as life expectancy, education metrics, access to electricity, and GDP per capita?*

### Rationale  
This question examines whether a nation’s financial strength, reflected through its total reserves, is associated with improvements in key social well-being indicators. By analyzing relationships with variables such as life expectancy, education, access to electricity, and GDP per capita, I aim to understand whether economic capacity translates into tangible social progress. The rationale behind this question is to explore how financial stability can influence quality of life and whether countries with stronger reserve positions demonstrate more equitable and sustained development outcomes.


### Task Abstraction: 
- **Correlate**: What are the relationships between total reserves and social well-being indicators?
- **Compute Derived Value**: What is the average years in school (education index) in each country?
- **Compute Derived Value**: What is the blended score of countries consisting of social factors?
- **Retrieve Value**: What percentage of countries have similar education metrics between men and women?
___

In [1]:
import pandas as pd
import altair as alt
import numpy as np

DATA_PATH = 'final_clean_merged_owid.csv' 
x = pd.read_csv(DATA_PATH, parse_dates=['year'])

___
## **Section 1 - Column Selection & Renaming**

To streamline analysis, the dataset is filtered to include only the variables relevant to the two analytic questions:
- Reserve-related attributes for computing *gold_share* (Q1).  
- Social and economic indicators for assessing relationships with total reserves (Q2).  

The resulting dataframe retains only 12 columns, balancing completeness with interpretability. The dataset has also been filtered to only include dates where majority of the gold reserve data is available (2005-2015)


In [2]:
relevant_cols = [
    'country_x',
    'iso_code',
    'year',
    'region',
    'income_group',
    'Total reserves (includes gold, current US$)',
    'Total reserves minus gold (current US$)',
    'income',
    'life_expectancy',
    'years_in_school_men',
    'years_in_school_women',
    'Access to electricity (% of population)'
]

df = x[relevant_cols]
df = df[df['year'].dt.year.between(2005,2015)]
df = df.rename(columns = {'country_x':'country',
                        'Total reserves (includes gold, current US$)':'reserves_gold', 
                        'Total reserves minus gold (current US$)':'reserves_no_gold',
                        'Access to electricity (% of population)':'elec_access',
                        })
print('Filtered dataset shape:', df.shape)
print('Columns retained:\n', list(df.columns))
df

Filtered dataset shape: (2413, 12)
Columns retained:
 ['country', 'iso_code', 'year', 'region', 'income_group', 'reserves_gold', 'reserves_no_gold', 'income', 'life_expectancy', 'years_in_school_men', 'years_in_school_women', 'elec_access']


,country,iso_code,year,region,income_group,reserves_gold,reserves_no_gold,income,life_expectancy,years_in_school_men,years_in_school_women,elec_access
15,Afghanistan,AFG,2005-01-01,Asia,Low,NaN,NaN,1140.0,53.9,3.27,0.70,28.7
16,Afghanistan,AFG,2006-01-01,Asia,Low,NaN,NaN,1160.0,54.1,3.35,0.72,33.5
17,Afghanistan,AFG,2007-01-01,Asia,Low,NaN,NaN,1290.0,54.6,3.43,0.75,38.4
18,Afghanistan,AFG,2008-01-01,Asia,Low,3.042274e+09,2.430836e+09,1300.0,55.2,3.51,0.78,42.4
19,Afghanistan,AFG,2009-01-01,Asia,Low,4.265889e+09,3.501371e+09,1530.0,55.7,3.59,0.80,48.3
...,...,...,...,...,...,...,...,...,...,...,...,...
7516,Zimbabwe,ZWE,2011-01-01,Africa,Low,6.602293e+08,6.591757e+08,1660.0,51.9,10.50,9.53,36.9
7517,Zimbabwe,ZWE,2012-01-01,Africa,Low,5.755528e+08,5.744077e+08,1850.0,54.1,10.70,9.70,44.0
7518,Zimbabwe,ZWE,2013-01-01,Africa,Low,4.749687e+08,4.744827e+08,1900.0,55.6,10.80,9.86,40.6
7519,Zimbabwe,ZWE,2014-01-01,Africa,Low,3.638328e+08,3.633461e+08,1910.0,57.0,10.90,10.00,32.3


___
## **Section 2 – Derived Columns & Missing Value Summary**

This section derives two new variables essential for the analytic questions:

- `gold_share` - the proportion of gold in a nation’s total reserves.  
  $$\text{gold\_share} = 
  \frac{(\text{Total reserves (includes gold)} - \text{Total reserves (without gold)})}
       {\text{Total reserves (includes gold)}} \times 100$$

- `education_index` - the average years of schooling between men and women.  
  $$\text{education\_index} = 
  \frac{\text{Years in school (men)} + \text{Years in school (women)}}{2}$$

Both derived columns enable meaningful cross-country and regional comparisons for subsequent analysis.




In [3]:
# gold_share
df['gold_share'] = ((df['reserves_gold'] - df['reserves_no_gold']) / df['reserves_gold']) * 100
df.loc[(df['gold_share'] < 0) | (df['gold_share'] > 100), 'gold_share'] = np.nan

# education_index
df['education_index'] = (df['years_in_school_men'] + df['years_in_school_women']) / 2

display(df[['gold_share', 'education_index']].describe().T)

# Removing rows with missing data
df = df[(~df['gold_share'].isna()) & (~df['education_index'].isna())]
df

,count,mean,std,min,25%,50%,75%,max
gold_share,1897.0,8.054371,16.475323,0.00,0.00000,1.22449,7.292245,90.507339
education_index,1852.0,9.516901,3.454543,1.65,6.88875,9.90250,12.500000,15.500000


,country,iso_code,year,region,income_group,reserves_gold,reserves_no_gold,income,life_expectancy,years_in_school_men,years_in_school_women,elec_access,gold_share,education_index
18,Afghanistan,AFG,2008-01-01,Asia,Low,3.042274e+09,2.430836e+09,1300.0,55.2,3.51,0.78,42.4,20.098073,2.145
19,Afghanistan,AFG,2009-01-01,Asia,Low,4.265889e+09,3.501371e+09,1530.0,55.7,3.59,0.80,48.3,17.921655,2.195
20,Afghanistan,AFG,2010-01-01,Asia,Low,5.162441e+09,4.174367e+09,1610.0,56.2,3.68,0.83,42.7,19.139657,2.255
21,Afghanistan,AFG,2011-01-01,Asia,Low,6.344642e+09,5.268341e+09,1660.0,56.7,3.76,0.86,43.2,16.963929,2.310
22,Afghanistan,AFG,2012-01-01,Asia,Low,7.152304e+09,5.982504e+09,1840.0,57.2,3.85,0.89,69.1,16.355571,2.370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7516,Zimbabwe,ZWE,2011-01-01,Africa,Low,6.602293e+08,6.591757e+08,1660.0,51.9,10.50,9.53,36.9,0.159578,10.015
7517,Zimbabwe,ZWE,2012-01-01,Africa,Low,5.755528e+08,5.744077e+08,1850.0,54.1,10.70,9.70,44.0,0.198957,10.200
7518,Zimbabwe,ZWE,2013-01-01,Africa,Low,4.749687e+08,4.744827e+08,1900.0,55.6,10.80,9.86,40.6,0.102334,10.330
7519,Zimbabwe,ZWE,2014-01-01,Africa,Low,3.638328e+08,3.633461e+08,1910.0,57.0,10.90,10.00,32.3,0.133760,10.450


In [4]:
print('\nDataset information:\n')
df.info()

# Missing Value Summary
print('\nPercentage of missing values per column:')
missing_pct = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
display(missing_pct.head(15))


Dataset information:

<class 'pandas.core.frame.DataFrame'>
Index: 1678 entries, 18 to 7520
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   country                1678 non-null   object        
 1   iso_code               1678 non-null   object        
 2   year                   1678 non-null   datetime64[ns]
 3   region                 1678 non-null   object        
 4   income_group           1678 non-null   object        
 5   reserves_gold          1678 non-null   float64       
 6   reserves_no_gold       1678 non-null   float64       
 7   income                 1678 non-null   float64       
 8   life_expectancy        1678 non-null   float64       
 9   years_in_school_men    1678 non-null   float64       
 10  years_in_school_women  1678 non-null   float64       
 11  elec_access            1676 non-null   float64       
 12  gold_share             1678 non-null   floa

elec_access              0.12
country                  0.00
year                     0.00
region                   0.00
income_group             0.00
iso_code                 0.00
reserves_gold            0.00
reserves_no_gold         0.00
life_expectancy          0.00
income                   0.00
years_in_school_men      0.00
years_in_school_women    0.00
gold_share               0.00
education_index          0.00
dtype: float64

___
## **Section 3 - Univariate Exploratory Data Analysis (EDA)**

This section summarizes the main quantitative and categorical attributes in the dataset.  
Univariate analysis helps reveal variable distributions, outliers, and potential data-quality issues before deeper comparisons.

The key attributes explored here are:
- `gold_share`
- `reserves_gold`
- `reserves_no_gold`
- `income` (GDP per capita)
- `education_index`
- `life_expectancy`
- `elec_access`
- `region`
- `income_group`


### Quantitative Variable Distributions

In [5]:

numeric_vars = [
    'gold_share',
    'reserves_gold',
    'reserves_no_gold',
    'income',
    'education_index',
    'life_expectancy',
    'elec_access'
]

display(df[[
    'gold_share',
    'reserves_gold',
    'reserves_no_gold',
    'income',
    'education_index',
    'life_expectancy',
    'elec_access'
    ]].describe().T)

for col in numeric_vars:
    chart = (
        alt.Chart(df).mark_bar().encode(
            alt.X(f'{col}:Q', bin = alt.Bin(maxbins = 50), title = col.replace('_', ' ').title()),
            alt.Y('count()')
        ).properties(
            width = 400, 
            height = 250, 
            title = f'Distribution of {col.replace('_', ' ').title()}'
        )
    )
    display(chart)


,count,mean,std,min,25%,50%,75%,max
gold_share,1678.0,8.430464e+00,1.692179e+01,0.00,0.000000e+00,1.798743e+00,7.593035e+00,9.050734e+01
reserves_gold,1678.0,6.060913e+10,2.543070e+11,23576837.33,1.002294e+09,5.271761e+09,3.185107e+10,3.900000e+12
reserves_no_gold,1678.0,5.445876e+10,2.481796e+11,23576837.33,9.482026e+08,4.591295e+09,2.828233e+10,3.860000e+12
income,1678.0,1.795065e+04,1.945704e+04,598.00,4.057500e+03,1.140000e+04,2.540000e+04,1.290000e+05
education_index,1678.0,9.835858e+00,3.218184e+00,2.07,7.376250e+00,1.010250e+01,1.270000e+01,1.550000e+01
life_expectancy,1678.0,7.121937e+01,8.533803e+00,32.10,6.570000e+01,7.350000e+01,7.750000e+01,8.380000e+01
elec_access,1676.0,8.008634e+01,2.957659e+01,1.30,6.700000e+01,9.810000e+01,1.000000e+02,1.000000e+02


alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

#### Observations

The distribution of `gold_share` is heavily right-skewed, with the majority of countries holding less than 10% of their total reserves in gold. Only a small number of nations maintain significantly higher proportions, indicating that gold plays a limited but strategic role in most reserve portfolios.  
Similarly, the distribution of `income` (GDP per capita) is highly right-skewed, showing that only a few countries reach very high income levels, while most remain in lower income brackets.  
The `education_index` displays a moderate left skew, suggesting that most countries have relatively high average years of schooling, though some still exhibit limited educational attainment. However, it is possible that this result is skewed by the fact that low income countries will generally have higher values for male school attendance counts. It will be interesting to look at how our answers to the analytical question changes by gender later on.  
`life_expectancy` follows a similar trend, clustering between 70 and 80 years, which reflects global improvements in healthcare and living conditions.  
In contrast, `elec_access` peaks sharply near 100%, signifying that most countries have achieved near-universal electricity access, with only a few still facing infrastructure challenges.  
Overall, these distributions reveal clear global inequalities: economic and financial strength remain concentrated among a few high-income nations, while social indicators such as education, life expectancy, and electricity access show more consistent progress across regions.


Looking at these charts we can see that `reserves_gold` and `reserves_no_gold` have extremely large ranges as some countries have reserves in the millions while others have in the trillions. This means that our distribution graph becomes skewed. In this case, there are a lot more countries with small values rather than large values leading to a right-skew. To combat this I will apply a log function so that we can view the distribution more clearly without a skew.

In [6]:
df['log_reserves_gold'] = np.log1p(df['reserves_gold'])
df['log_reserves_no_gold'] = np.log1p(df['reserves_no_gold'])


/tmp/ipykernel_193/2621957894.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['log_reserves_gold'] = np.log1p(df['reserves_gold'])
/tmp/ipykernel_193/2621957894.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['log_reserves_no_gold'] = np.log1p(df['reserves_no_gold'])


In [7]:
display(df[['log_reserves_gold']].describe().T)
alt.Chart(df).mark_bar().encode(
    alt.X('log_reserves_gold:Q', bin = alt.BinParams(maxbins=50)),
    alt.Y('count()')
)


,count,mean,std,min,25%,50%,75%,max
log_reserves_gold,1678.0,22.475634,2.276739,16.975775,20.725558,22.38563,24.184336,28.991998


alt.Chart(...)

In [8]:
display(df[['log_reserves_no_gold',]].describe().T)
alt.Chart(df).mark_bar().encode(
    alt.X('log_reserves_no_gold:Q', bin = alt.BinParams(maxbins=50)),
    alt.Y('count()')
)


,count,mean,std,min,25%,50%,75%,max
log_reserves_no_gold,1678.0,22.355928,2.229562,16.975775,20.670079,22.247414,24.065503,28.981688


alt.Chart(...)

The log-transformed distributions of both `reserves_gold` and `reserves_no_gold` appear roughly symmetric and approximately bell-shaped, indicating that the logarithmic scale successfully reduces the extreme right-skew seen in the original data. On the linear scale, a handful of very large economies dominated the visualization, compressing most countries near zero. After applying the logarithmic transformation, the values are more evenly spread, revealing that most nations cluster within a similar magnitude of reserve levels, while only a few remain at the upper end of the range. This transformation highlights a more balanced global reserve distribution and allows meaningful comparisons between countries of different economic sizes.


### Categorical Variable Distributions

To complement the numeric summaries, the categorical variables `region` and `income_group` are examined to understand the composition of the dataset.  
These plots show how many observations come from each region and economic group, helping identify any data imbalance that may influence later comparisons.


In [9]:

region_chart = (
    alt.Chart(df).mark_bar().encode(
        alt.X('region:N', sort='-y', title='Region'),
        alt.Y('count()', title='Number of Records'),
        tooltip=['region', 'count()']
    ).properties(
        width=400,
        height=350, 
        title='Distribution of Records by Region'
    )
)


income_group_chart = (
    alt.Chart(df).mark_bar(color='orange').encode(
        alt.X('income_group:O', sort=['Low income', 'Lower middle income', 'Upper middle income', 'High income'], title = 'Income Group'),
        alt.Y('count()', title='Number of Records'),
        tooltip=['income_group', 'count()']
    ).properties(
        width=400,
        height=350, 
        title='Distribution of Records by Income Group'
    )
)

region_chart | income_group_chart


alt.HConcatChart(...)

#### Observations

The distribution of records by `region` shows that Asia and Africa contribute the largest number of entries, followed closely by Europe and the Americas. Oceania is the least represented region, which may slightly limit global balance in the dataset.  
The `income_group` distribution indicates that upper-middle- and high-income countries make up the majority of observations, while low-income economies are less represented. This imbalance suggests that wealthier nations have more consistent and comprehensive reporting of reserve and social indicators.  
Together, these distributions reveal that while the dataset captures a wide global spread, it is somewhat weighted toward economically stronger and more data-rich regions.


___
## **Section 4 - Multivariate Exploratory Data Analysis (EDA)**

### Analytic Question 1: 
*How does the proportion of gold in total reserves differ across countries and regions, and what is the relationship between gold-share and income groups?*

This section investigates variation in `gold_share` (the proportion of total reserves held as gold) across `region` and `income_group`.  
Understanding these patterns helps reveal whether economic status or geographic location influences how nations manage gold within their reserve portfolios.



In [10]:

alt.Chart(df).mark_bar().encode(
    alt.Color('region:N', title='Region'),
    alt.X('gold_share:Q', bin = alt.BinParams(maxbins = 30), title='Gold Share (%)'),
    alt.Y('count()', stack = 'normalize'),
    tooltip=['region', 'count()']
).properties(
    width=400, 
    height=300, 
    title='Gold Share Distribution by Region')



alt.Chart(...)

In [11]:
alt.Chart(df).mark_bar().encode(
    alt.Color('income_group:O', sort=['Low income', 'Lower middle income', 'Upper middle income', 'High income'], title='Income Group'),
    alt.X('gold_share:Q', bin = alt.BinParams(maxbins = 30), title='Gold Share (%)'),
    alt.Y('count()', stack = 'normalize'),
    tooltip=['income_group', 'count()']
).properties(
    width=400, 
    height=300, 
    title='Gold Share Distribution by Income Group')

alt.Chart(...)

#### Observations

The stacked normalized distributions show distinct patterns in how gold reserves are held across both `region` and `income_group`.  
Europe and Asia display the strongest representation in higher gold-share categories, while Africa and the Americas are more concentrated in lower ranges.  
When viewed by `income_group`, high-income economies dominate the higher gold-share bins, indicating that wealthier nations tend to maintain larger proportions of gold in their reserves.  
In contrast, low- and lower-middle-income countries are concentrated at the lower end of the gold-share scale, highlighting that economic capacity and reserve diversification strategies likely influence gold reliance.



### Analytic Question 2:
*To what extent does a nation’s total reserves relate to social well-being indicators such as life expectancy, education metrics, access to electricity, and GDP per capita?*  

This section explores whether higher levels of total reserves (`reserves_gold`) are associated with stronger social and economic indicators.  
By examining these relationships, we can assess if reserve capacity translates into improvements in citizens’ quality of life.


In [12]:
cols_corr = [
    'log_reserves_gold',
    'income',
    'education_index',
    'life_expectancy',
    'elec_access'
]
grouped = df.groupby('country')[cols_corr].mean()
corr_matrix = grouped.corr()
cor_data = corr_matrix.stack().reset_index()
cor_data = cor_data.rename(columns={0: 'correlation',
                                    'level_0': 'variable1',
                                    'level_1': 'variable2'})
cor_data['correlation_label'] = cor_data['correlation'].map('{:.2f}'.format)
display(cor_data)

alt.Chart(cor_data).mark_rect().encode(
    alt.X('variable1:O'),
    alt.Y('variable2:O'),
    alt.Color('correlation').scale(scheme='reds'),
    alt.Tooltip(['correlation_label']),
).interactive().properties(width=500, height=500)


,variable1,variable2,correlation,correlation_label
0,log_reserves_gold,log_reserves_gold,1.000000,1.00
1,log_reserves_gold,income,0.417418,0.42
2,log_reserves_gold,education_index,0.411689,0.41
3,log_reserves_gold,life_expectancy,0.477365,0.48
4,log_reserves_gold,elec_access,0.481900,0.48
5,income,log_reserves_gold,0.417418,0.42
6,income,income,1.000000,1.00
7,income,education_index,0.565889,0.57
8,income,life_expectancy,0.609161,0.61
9,income,elec_access,0.497735,0.50


alt.Chart(...)

In [13]:


alt.Chart(grouped).mark_circle().encode(
    alt.X(alt.repeat('column'), type='quantitative'),
    alt.Y(alt.repeat('row'), type='quantitative'),
).properties(
    width=200,
    height=200
).repeat(
    row=cols_corr,
    column=cols_corr
)


alt.RepeatChart(...)

In [14]:

df['edu_gap'] = df['years_in_school_men'] - df['years_in_school_women']
similar_edu = df[df['edu_gap'].between(-0.3,0.3)]
percentage_similar = (len(similar_edu) / len(df)) * 100
print(f'{percentage_similar:.2f}% of countries have similar education metrics between men and women.\n')


17.52% of countries have similar education metrics between men and women.



/tmp/ipykernel_193/2781539248.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['edu_gap'] = df['years_in_school_men'] - df['years_in_school_women']


#### Observations

The multivariate analysis reveals consistent positive relationships between a nation’s financial reserves and its social development indicators.  
From the correlation matrix, `reserves_gold` and `income` show moderate correlations (approximately 0.4–0.5) with `life_expectancy`, `education_index`, and `elec_access`.  
This suggests that countries with greater reserve holdings and higher GDP per capita tend to experience longer life expectancy, improved education outcomes, and broader access to electricity.  
The strongest associations emerge between `education_index`, `life_expectancy`, and `elec_access`, indicating that social progress often advances in parallel across health, education, and infrastructure dimensions.  

The pairwise scatterplots provide further evidence of these relationships.  
While clear upward trends are visible between economic and social variables, the dispersion across countries reveals that financial wealth does not guarantee uniform social outcomes.  
A few countries display high reserves with relatively low life expectancy or education levels, implying that national wealth may not always translate into  social investment.  
Similarly, smaller economies with focused social policies sometimes achieve disproportionately strong outcomes despite limited reserves. 

Notably, the relationship between total reserves and GDP per capita appears exponential rather than linear, with high-income countries dominating the upper-right region of the scatterplots.  
This reflects the snowball-effect of 'the rich getting richer'.
Conversely, the cluster of low- and middle-income nations near the lower end of the scale highlights the persistent developmental divide in the global economy.  

The retrieve-value analysis shows that only **17.52%** of countries exhibit near-equal years of schooling between men and women, revealing that gender disparities remain significant despite global improvements in education access.  
This finding aligns with the broader interpretation that economic and infrastructural growth often outpaces social equality, emphasizing the need for more inclusive policy frameworks.  

Overall, the multivariate evidence displays a broad but uneven alignment between economic strength and social well-being.  
Countries with larger wallets tend to invest more effectively in human capital and infrastructure, yet the variability across countries illustrates that financial stability alone does not ensure equitable social advancement.  


___
## **Section 5 – Summary and Key Insights**

This project explored how nations structure their reserves and how financial capacity relates to broader social development outcomes.  
The analyses centered on two key questions:  
1. How does the proportion of gold in total reserves differ across countries and regions, and what is the relationship between gold-share and income groups?  
2. To what extent does a nation’s total reserves relate to social well-being indicators such as life expectancy, education metrics, access to electricity, and GDP per capita?  

**Key Insights**

1. **Gold’s role is regionally and economically stratified.**  
   The gold-share analysis revealed that wealthier countries and those in Europe and Asia maintain higher proportions of gold in their total reserves.  
   This reflects long-standing central-bank practices and the greater financial flexibility of high-income economies to diversify into less liquid assets like gold.  
   In contrast, lower-income nations and those in Africa and the Americas typically rely more on foreign currencies for liquidity and short-term security.  

2. **Economic capacity strongly aligns with social advancement, but not uniformly.**  
   The multivariate exploration of total reserves against life expectancy, education, electricity access, and income indicated strong positive correlations across all variables.  
   Countries with higher reserves and income levels generally enjoy better health, education, and infrastructure outcomes.  
   However, scatterplots showed significant variance within income groups, underscoring that financial strength alone does not ensure equal social outcomes.  
   Policy efficiency, governance quality, and targeted investments play critical roles in translating economic strength into tangible improvements in well-being.  

3. **Gender disparities in education remain persistent.**  
   The retrieve-value analysis showed that only about **17.5%** of countries achieve near-equal schooling years between men and women.  
   Although global education levels have risen overall, this highlights that gender equality in human capital formation still lags behind (as of 2015) broader social and economic progress.  

4. **Global imbalance.**  
   While many nations exhibit upward trends in both reserves and development indicators, the concentration of high reserve holdings and gold dependence among a small group of advanced economies points to continuing global imbalances in economic resilience and social opportunity.  

**Overall Conclusion**

The findings suggest that financial strength and social well-being are deeply interconnected yet unevenly distributed.  
Gold’s prominence within reserve portfolios serves as both a symbol and a practical tool of economic maturity, a privilege of nations with economic stability.  
Meanwhile, the broader links between reserves and well-being emphasizes that economic growth must be coupled with inclusive policy frameworks to convert wealth into equitable social outcomes.


___
## **Section 6 - Sketches**

### Analytical Question 1:
#### Lo-Fi 1 
![Lo-Fi 1 Sketch](lofi_1.1.jpg)

For my first sketch, I experimented with violin plots to show how gold share is distributed across income groups, faceted by region. I like that this design shows variation and density at the same time because it gives a sense of how uneven gold reliance can be between income levels. Even though the drawing is a bit messy, the idea feels strong conceptually. If I actually built this, I would probably add median markers or clear cut colors so the overlapping shapes are easier to read, since right now it could look confusing for people who are not used to violin plots. Additionally a problem may arise due to `gold_share`'s distribution being skewed. It might be a better idea to take the log of gold_share to create the graph so that our plot actually resembles a violin rather than a lab flask. I also think that I am wasting the temporal data available and so I will add a slider in encoded on years
| **Channel** | **Encoding** |
|--------------|--------------|
| X-axis | Region (faceted into multiple panels) |
| Y-axis | Gold share (%) |
| Color | Income group |
| Tooltip | Country, region, income group, gold share |



#### Lo-Fi 2  
![Lo-Fi 2 Sketch](lofi_1.2.jpg)
This one focuses on trends over time, plotting gold share by region from 2005 to 2015. I like that it is straightforward and makes regional shifts easy to compare since you can immediately see which areas are increasing or declining. It is simple and clean, which fits the story well. If I polished it, I would probably move the region labels next to the line ends instead of using a legend to make it easier to follow each line without jumping back and forth.
| **Channel** | **Encoding** |
|--------------|--------------|
| X-axis | Year |
| Y-axis | Gold share (%) |
| Color | Region |
| Tooltip | Region, year, average gold share |
| Line | Temporal trend for each region |

#### Lo-Fi 3  
![Lo-Fi 3 Sketch](lofi_1.3.jpg)
The last sketch is a choropleth map showing how gold share varies across the world using the geoshape mark. This is probably my favorite because it visually captures the global scope of the question and you can instantly tell which regions hold more gold in their reserves. The red gradient gets the point across, but I would change it to a gold-to-beige palette to make it feel more relevant and less aggressive. Even though the shapes are rough, the concept of combining color intensity with geography feels the most powerful and presentation-ready. Once again though, I feel like I wasted the temporal data and so the slider will also be there for this vizualization so that users can look through the years of data. 
| **Channel** | **Encoding** |
|--------------|--------------|
| Geographic projection | World map (by region or country) |
| Color hue | Gold share (%) from low to high |
| Tooltip | Region or country, gold share |
| Legend | Color gradient scale for gold share |
| Shape | Geographic boundaries of regions or countries |

### Hi-Fi Sketch:
![Hi-Fi 1 Sketch](hifi_1.1.jpg)
![Hi-Fi 2 Sketch](hifi_1.2.jpg)

### Analytical Question 2:
#### Lo-Fi 1 
![Lo-Fi 1 Sketch](lofi_2.1.jpg)
This design shows a scatterplot with a regression line connecting total reserves (log) and a blended score of social well-being indicators. I like that it simplifies several social metrics into one score, which makes the relationship between reserves and social progress easier to interpret. The trendline gives a clear sense of direction even if the points are dispersed. If I improved this, I would add slight transparency to the points to handle overlap and maybe highlight outlier countries to make the story more engaging. Overall, it is a clean and analytical design that directly addresses the question.
 **Channel** | **Encoding** |
|--------------|--------------|
| X-axis | Blended score of social factors (0–1) |
| Y-axis | Total reserves (log) |
| Color | Region |
| Tooltip | Country, region, reserves, social score |
| Trendline | Linear regression fit to visualize correlation |

#### Lo-Fi 2 
![Lo-Fi 1 Sketch](lofi_2.2.jpg)
This bubble chart compares life expectancy and total reserves, with GDP per capita represented through bubble size and region as color. I like how it adds another variable without making the chart overwhelming. It communicates multiple dimensions clearly and shows how wealthier countries generally have higher reserves and longer life expectancy. If I adjusted anything, I would make the bubbles slightly transparent and include a reference grid to help compare clusters across regions. It is visually engaging and captures both scale and inequality well.  

| **Channel** | **Encoding** |
|--------------|--------------|
| X-axis | Life expectancy |
| Y-axis | Total reserves (log) |
| Size | GDP per capita |
| Color | Region |
| Tooltip | Country, region, income, life expectancy, reserves |

#### Lo-Fi 3
![Lo-Fi 1 Sketch](lofi_2.3.jpg)
This grouped bar chart bins total reserves into categories and compares the average blended social factor score for each region within those bins. I like that it converts continuous data into understandable segments that make regional comparison straightforward. The consistent color palette makes the differences clear. If I changed anything, I would ensure the bins are based on real quantiles instead of arbitrary categories and possibly order them by average score for smoother comparison. It works very well for highlighting categorical differences and complements the other charts by showing a higher-level summary.  

| **Channel** | **Encoding** |
|--------------|--------------|
| X-axis | Total reserves (binned: Less, Millions, Billions, Trillions) |
| Y-axis | Average blended social factor score |
| Color | Region |
| Tooltip | Region, reserve bin, mean social score |
| Grouping | Bars grouped by reserve bins within each region |

### HiFi Sketch
![Hi-Fi 1 Sketch](hifi_2.jpg)